[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DhiSys-AI/MRM-Trace/blob/main/notebooks/002%20-%20YAML%20Config%20%26%20Experiment%20Planning.ipynb)

# mrm-trace · 002: YAML Config & Experiment Planning

Every mrm-trace experiment is described by a single YAML file. This notebook walks through
the full schema, shows how validation catches errors before any work starts, and demonstrates
how a compact sweep config expands into a concrete list of runs.

> **This notebook runs entirely without model files or root access.**
> It uses the Python API with YAML strings and temp files.

| Section | What you'll learn |
|---|---|
| 1. Install | Install mrm-trace |
| 2. YAML anatomy | The 9 config sections |
| 3. Write & load | `Experiment.from_yaml()` |
| 4. Validation | Schema errors, path checks, environment warnings |
| 5. Sweep expansion | How model × context × reps becomes a run list |
| 6. Multi-model sweeps | Compare two models in one experiment |
| 7. Collector & analyser tuning | perf_mem vs memray, metric selection |
| 8. Next steps | Connecting config → collection |

## 1. Install & Verify

In [ ]:
!pip install -q mrm-trace

In [ ]:
import mrm_trace
print(f"mrm-trace {mrm_trace.__version__}")

## 2. Experiment YAML Anatomy

A config has **9 top-level sections**. Only `experiment` and `models` are required.
Everything else uses sensible defaults.

| Section | Required | Purpose |
|---|---|---|
| `experiment` | ✅ | Name, description, version, authors |
| `models` | ✅ | List of models (engine, path/hf_repo, family, params) |
| `sweep` | | Context lengths, batch sizes, output tokens, repetitions |
| `prompts` | | Prompt source file (JSONL) |
| `collector` | | perf_mem / memray / process_monitor settings |
| `telemetry` | | Hardware + thermal capture settings |
| `labeller` | | Symbol classification rules |
| `analyser` | | Metrics to compute and grouping |
| `output` | | Results directory, format, compression |

### Model engines

| Engine | Required field | Notes |
|---|---|---|
| `llamacpp` | `path` (GGUF file) | Runs locally |
| `vllm` | `hf_repo` or `path` | Requires GPU, supports HuggingFace |

### Collector hierarchy

| Priority | Collector | Captures | Root needed? |
|---|---|---|---|
| Primary | `perf_mem` | Hardware loads/stores, cache level, NUMA, latency | Yes (Linux PMU) |
| Fallback | `memray` | Python-level allocations (malloc/free) | No |
| Always on | `process_monitor` | RSS / VMS / CPU at coarse intervals | No |

## 3. Writing & Loading a Config

The minimum valid config needs `experiment.name` and at least one model.
Here we write a YAML string to a temp file, then load it with the Python API.

In [ ]:
import pathlib, textwrap

config_yaml = textwrap.dedent('''
    experiment:
      name: "memory_retention_baseline"
      description: "Weight vs KV-cache retention characterisation"
      version: "1.0.0"
      date_created: "2026-05-23"
      authors: ["Your Name"]

    models:
      - id: "tinyllama-1b"
        engine: "llamacpp"
        path: "/content/models/tinyllama-1b-q4.gguf"
        family: "llama"
        params_billions: 1.1
        quantisation: "Q4_K_M"

    sweep:
      context_lengths: [512, 2048]
      batch_sizes: [1]
      output_tokens: [128]
      repetitions: 2
      warmup_runs: 1
''').strip()

config_path = pathlib.Path("/content/experiment.yaml")
config_path.write_text(config_yaml)
print("Wrote:", config_path)
print()
print(config_yaml)

In [ ]:
from mrm_trace.api import Experiment

exp = Experiment.from_yaml(config_path)
cfg = exp.config

print("Loaded successfully!")
print(f"  Experiment : {cfg.experiment.name}")
print(f"  Models     : {[m.id for m in cfg.models]}")
print(f"  Contexts   : {cfg.sweep.context_lengths}")
print(f"  Repetitions: {cfg.sweep.repetitions}")
print(f"  Warmup runs: {cfg.sweep.warmup_runs}")
print(f"  Compression: {cfg.output.compression.value}")
print(f"  Analyser   : {[m.value for m in cfg.analyser.metrics]}")

## 4. Validation

mrm-trace validates configs in three passes:

| Pass | What it checks |
|---|---|
| **Schema** (Pydantic) | Types, required fields, enum values — runs at load time |
| **Semantics** | Duplicate model IDs, very large sweep warnings |
| **Model paths** | GGUF / checkpoint files exist on disk |
| **Environment** | `perf` in PATH, `memray` installed |

Schema errors are fatal (raised immediately). The other three return `ValidationResult` objects with `.errors` and `.warnings`.

In [ ]:
from pydantic import ValidationError
from mrm_trace.config.schema import ExperimentConfig

# Test 1: missing models
print("Test 1: missing 'models' field")
try:
    ExperimentConfig.model_validate({"experiment": {"name": "test"}})
except (ValidationError, ValueError) as e:
    print(f"  Error: {str(e).splitlines()[0]}")
print()

# Test 2: llamacpp model without a path
print("Test 2: llamacpp model without 'path'")
try:
    ExperimentConfig.model_validate({
        "experiment": {"name": "test"},
        "models": [{"id": "m1", "engine": "llamacpp"}],
    })
except (ValidationError, ValueError) as e:
    for line in str(e).splitlines()[:4]:
        print(f"  {line}")
print()

# Test 3: empty context_lengths
print("Test 3: empty context_lengths")
try:
    ExperimentConfig.model_validate({
        "experiment": {"name": "test"},
        "models": [{"id": "m1", "engine": "llamacpp", "path": "/m.gguf"}],
        "sweep": {"context_lengths": []},
    })
except (ValidationError, ValueError) as e:
    print(f"  Error: {str(e).splitlines()[0]}")

In [ ]:
from mrm_trace.config.validators import (
    validate_config_semantics,
    validate_model_paths,
    validate_environment,
)

def run_all_checks(experiment: Experiment) -> None:
    checks = [
        ("semantics",   validate_config_semantics),
        ("model paths", validate_model_paths),
        ("environment", validate_environment),
    ]
    any_error = False
    for label, fn in checks:
        result = fn(experiment.config)
        for err in result.errors:
            print(f"  [ERROR] ({label}) {err}")
            any_error = True
        for warn in result.warnings:
            print(f"  [WARN]  ({label}) {warn}")
    if not any_error:
        print("  No errors.")

print("Validation output for our config:")
run_all_checks(exp)
print()
print("Notes:")
print("  Model path [ERROR] is expected — no .gguf file on Colab.")
print("  Path errors are informational: collection fails at run time, not at load time.")
print("  perf [WARN] is expected on Colab — perf is not installed by default.")

## 5. Sweep Expansion & Run Matrix

`exp.plan()` materialises the full list of `RunConfig` objects from the sweep parameters.

**Expansion formula:**
```
total = n_models × n_contexts × n_batch_sizes × n_output_token_counts × repetitions
      + (n_models × n_contexts × ... × warmup_runs)
```

Warmup runs come first and are **excluded from aggregated metrics**.

**Run ID format:** `<model_id>-c<ctx>-b<batch>-t<tokens>-r<rep>-<timestamp>`

Run IDs are deterministic and encode the full configuration, making results self-describing.

In [ ]:
runs = exp.plan()

warmups = [r for r in runs if r.is_warmup]
actual  = [r for r in runs if not r.is_warmup]

cfg = exp.config
print(f"Sweep:")
print(f"  {len(cfg.models)} model(s) × {len(cfg.sweep.context_lengths)} context(s) {cfg.sweep.context_lengths}")
print(f"  × {len(cfg.sweep.batch_sizes)} batch size(s) × {cfg.sweep.repetitions} rep(s)")
print(f"  + {cfg.sweep.warmup_runs} warmup run(s)")
print()
print(f"Total runs : {len(runs)}")
print(f"  Warmup   : {len(warmups)}")
print(f"  Actual   : {len(actual)}")

In [ ]:
import pandas as pd

df = pd.DataFrame([{
    "type"       : "warmup" if r.is_warmup else "actual",
    "model"      : r.model.id,
    "context"    : r.context_length,
    "batch"      : r.batch_size,
    "out_tokens" : r.output_tokens,
    "rep"        : r.rep_index,
    "run_id"     : r.run_id,
} for r in runs])

print("Full run matrix:")
display(df[["type", "model", "context", "batch", "out_tokens", "rep", "run_id"]])

## 6. Multi-Model Sweeps

List multiple models under `models:` to benchmark them side-by-side.
The same sweep parameters (context, batch, reps) apply to each model automatically.

In [ ]:
multi_yaml = textwrap.dedent('''
    experiment:
      name: "llama3_vs_tinyllama"
      description: "Compare 8B vs 1B on the same prompt suite"

    models:
      - id: "llama3-8b-q4"
        engine: "llamacpp"
        path: "/content/models/llama-3-8b.Q4_K_M.gguf"
        family: "llama3"
        params_billions: 8
        quantisation: "Q4_K_M"

      - id: "tinyllama-1b-q4"
        engine: "llamacpp"
        path: "/content/models/tinyllama-1b-q4.gguf"
        family: "llama"
        params_billions: 1.1
        quantisation: "Q4_K_M"

    sweep:
      context_lengths: [512, 2048, 4096]
      batch_sizes: [1]
      output_tokens: [128]
      repetitions: 3
      warmup_runs: 1
''').strip()

path2 = pathlib.Path("/content/multi_model.yaml")
path2.write_text(multi_yaml)

exp2    = Experiment.from_yaml(path2)
runs2   = exp2.plan()
actual2 = [r for r in runs2 if not r.is_warmup]

print(f"Models : {[m.id for m in exp2.config.models]}")
print(f"Contexts: {exp2.config.sweep.context_lengths}")
print(f"Total actual runs: {len(actual2)}")
print(f"  = {len(exp2.config.models)} models")
print(f"    × {len(exp2.config.sweep.context_lengths)} contexts")
print(f"    × {exp2.config.sweep.repetitions} reps")
print()

df2 = pd.DataFrame([{"model": r.model.id, "context": r.context_length, "rep": r.rep_index}
                    for r in actual2])
display(df2.groupby(["model", "context"]).size().rename("n_runs").reset_index())

## 7. Collector & Analyser Tuning

### Collector key settings

| Key | Purpose | Default |
|---|---|---|
| `collector.primary` | First collector to use | `perf_mem` |
| `collector.fallback` | Used if primary unavailable | `memray` |
| `perf_mem.ldlat_threshold` | Min load latency to capture (cycles) | `20` |
| `perf_mem.sample_period` | Sample 1 in N events | `4000` |
| `memray.native_traces` | Capture C-level allocs (requires root) | `true` |

### Analyser metric names

| Metric | What it measures |
|---|---|
| `retention_duration` | First-write → last-read lifetime per region |
| `write_once_ratio` | Fraction of addresses written exactly once |
| `read_frequency` | Read / write counts and read-per-write ratio |
| `working_set_size` | Unique pages touched per region |
| `access_locality` | Stride distribution, same-page fraction |
| `inter_access_interval` | Time between consecutive accesses (nanoseconds) |

In [ ]:
# Config with memray as primary (no root needed)
memray_yaml = textwrap.dedent('''
    experiment:
      name: "memray_no_root"

    models:
      - id: "tinyllama-1b"
        engine: "llamacpp"
        path: "/content/models/tinyllama-1b-q4.gguf"
        family: "llama"
        params_billions: 1.1
        quantisation: "Q4_K_M"

    collector:
      primary: "memray"           # Python-level allocations, no sudo needed
      fallback: "process_monitor"
      memray:
        native_traces: false        # false = Python only (no root); true = C allocs (root)
        follow_fork: true

    analyser:
      metrics:
        - retention_duration        # Only compute what we need
        - write_once_ratio
''').strip()

path3 = pathlib.Path("/content/memray_config.yaml")
path3.write_text(memray_yaml)
exp3 = Experiment.from_yaml(path3)

c = exp3.config.collector
print("Collector:")
print(f"  primary         : {c.primary.value}")
print(f"  fallback        : {c.fallback.value}")
print(f"  memray.native   : {c.memray.native_traces}")
print(f"  memray.fork     : {c.memray.follow_fork}")
print()
print("Analyser metrics:")
for m in exp3.config.analyser.metrics:
    print(f"  - {m.value}")

## 8. Next Steps

| Goal | How |
|---|---|
| Validate your config | `mrm-trace validate --config your_config.yaml` |
| Preview run list | `mrm-trace plan --config your_config.yaml` |
| Collect traces (Linux + sudo) | `mrm-trace run --config your_config.yaml` |
| See analysis output | [001 - Getting Started](001%20-%20Getting%20Started.ipynb) |
| See real memray collection | [003 - Real Collection Walkthrough](003%20-%20Real%20Collection%20Walkthrough.ipynb) |

---
*mrm-trace v0.1.2 · DhiSys AI*